
# Do LLMs know the difference between a pet chicken and a roast chicken?

## Word sense disambiguation in computational models and humans


In human language, words do not always have a fixed meaning. The most striking example is homonymous words: words that have the same form, but very different meanings. For instance, the word "bank", which has a different meaning in the context "I went to the bank to get some money" and "At the river bank, I met my old friend". Polysemous words are words that have different -- yet related -- meanings: for example, "chicken" is the same 'entity' in "My pet chicken is lovely" and "I am having roast chicken for dinner", but has very different meanings in these two contexts. In general, context can modulate almost any word's meaning. This poses a challenge in computational linguistics, as we need to find a way to differentiate among different meanings like humans do. Much research, resources, and models have been put forward to help with this challenge.

In this assignment, you are going to focus on [Trott and Bergen's (2021)](https://aclanthology.org/2021.acl-long.550/) RAW-C dataset: you are going to conduct a number of explorations with this dataset and partially replicate their research by the end of the assignment. In short, the authors explore how good LLMs are at capturing same/different meanings of words across contexts by comparing it to human judgements. To better understand the idea and the research, start by reading the paper.

This assignment entails a series of (interconnected) tasks (altogether worth 95 points):

* **Task 1**. Compute contextual word embeddings at different layers from Trott & Bergen's dataset. Here, each word is found in 4 sentences: 2 with one meaning, 2 with another meaning.
* **Task 2**. Compute sense embeddings for words in Trott & Bergen's dataset using WordNet, so you have an embedding for each definition of the word.
* **Task 3**. Compute the similarity between the contextual word embeddings of the homonyms at different layers and their sense embeddings; explore the relationship between homonyms and dominant senses quantitatively and qualitatively
* **Task 4**. Replicate part of Trott & Bergen's work by computing similarities across sentences with same/different meanings at the different layers and correlate with human similarities; visualise the results and reflect on them

In order to better understand the assignment, we recommend going through it all before starting so that it is clear how each part is connected to the next (which will help you make decisions about data structures, for instance).

# Task 1: Compute contextual word embeddings for homonyms [20 points]

## Task 1.1: read, explore and extract the necessary data [5 points]

First, you will have to (fork and) clone the github repository that stores the data you'll need. This can be found here: https://github.com/sashakenjeeva/raw-c . The repo also includes a README with a description of the original files in the repository, as well as some notes relevant for this assignment specifically.

In [1]:
#your code here (you can use as many cells as necessary/you prefer)

Make sure you mount the drive now so that you have access to the folder (think about setting the working directory in a way that is convenient).

In [2]:
# mount the drive here
!git clone https://github.com/DanielleGroenewegen/ChickenHotWings.git


fatal: destination path 'ChickenHotWings' already exists and is not an empty directory.


Now, you will have to read the data and organise it in a structure that works for the next parts of the assignment.

Read and explore the dataframe to see its structure (print part of it). What we need from it are the homonyms (in the form that they appear in the sentence -- the lexeme -- and in their regular form -- the lemma) and their corresponding sentences with different meanings (M1_a and M1_b have same meaning; M2_a, M2_b have same meaning). We only will need the stimuli that are in the final RAW-C dataset, as this is what we'll replicate at the end.

You can decide which data structure to use, but make sure that all these pieces of information are there (the word, the string, the meaning id, and the corresponding sentences) and easy to retrieve. Show your data at the end, as well as how many stimuli you end up with.

In [3]:
# imports

#general
import pandas as pd
import numpy as np
import pickle
import torch

#nltk
from nltk.corpus import wordnet as wn
import nltk

#bert
from transformers import BertTokenizer, BertModel


In [4]:
# read the data here
sunflower = pd.read_csv("/content/ChickenHotWings/data/stims/stimuli.csv")
#print(sunflower.head())
data = sunflower[['String', 'Word','M1_a', 'M1_b', 'M2_a', 'M2_b']].copy()

# remember to print the data structure you produce and the number of stimuli it contains!
print(data.head(),"\n")
#print(data.tail())
print(f'Number of stimuli: {len(data)}')

      String       Word                            M1_a  \
0       lamb       lamb  They liked the marinated lamb.   
1       book       book     He had a best-selling book.   
2  breakfast  breakfast   They ate a pancake breakfast.   
3    chicken    chicken   They liked the juicy chicken.   
4      class      class      It was a perceptive class.   

                               M1_b                              M2_a  \
0      They liked the grilled lamb.         They liked the cute lamb.   
1            He had a popular book.              He had a heavy book.   
2  They ate a nutritious breakfast.      They ate a family breakfast.   
3   They liked the roasted chicken.  They liked the clucking chicken.   
4       It was a misbehaving class.            It was a boring class.   

                            M2_b  
0  They liked the friendly lamb.  
1   He had a leather-bound book.  
2   They ate a lonely breakfast.  
3    They liked the pet chicken.  
4    It was a stimulating class

## Task 1.2: Compute the contextualised word embeddings [15 points]


Now that you have the homonyms and their corresponding sentences, we will need to compute word embeddings for each of them. For this we will use the BERT base model, in its uncased version.

That is, for each homonym, you will have to compute four embeddings: one for the homonym in M1_a, one in M1_b, one in M2_a, one in M2_b. However, we also want to look into different layers of the BERT model to see which one captures the homonym's meaning best: you want to calculate embeddings at the static layer and at layers 4, 8, 12.

We will use the package psycho-embeddings (you will use it in class), which allows us to specify which target words we want to obtain the embeddings of, in which sentences, and at which layers, among other things. Make sure to read the documentation of the package so that you know the meaning of the arguments and which ones will come useful to you.

First of all, install the psycho-embeddings package below.

Now, import the relevant module/function from psycho-embeddings and load the required BERT model.

In [5]:
# install the psycho-embeddings package here

!git clone https://github.com/MilaNLProc/psycho-embeddings.git
%cd /content/psycho-embeddings
!pip install -e . #current folder
from psycho_embeddings import ContextualizedEmbedder


fatal: destination path 'psycho-embeddings' already exists and is not an empty directory.
/content/psycho-embeddings
Obtaining file:///content/psycho-embeddings
  Preparing metadata (setup.py) ... done
  Attempting uninstall: psycho_embeddings
    Found existing installation: psycho_embeddings 0.1.0
    Uninstalling psycho_embeddings-0.1.0:
      Successfully uninstalled psycho_embeddings-0.1.0
  Running setup.py develop for psycho_embeddings


In [6]:
#your code here
model = ContextualizedEmbedder("bert-base-uncased", max_length=128)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/config.json
Model config BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/config.json
Model config BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ]

Now, test that everything works correctly by computing an embedding for the word "assignment" in the sentence "I am having so much fun with this assignment!", at static layer and layers 4, 8 and 12 (hint: think of tokenisation and how the embedder deals with that).

In [7]:
#your code here
embeddings = model.embed(
    words=["assignment"],
    target_texts=["I am having so much fun with this assignment!"],
    layers_id= [4, 8, 12],
    batch_size=1,
    return_static=True, #static layer (shown as layer -1)
)




Text tokenization:   0%|          | 0/1 [00:00<?, ? examples/s]

  0%|          | 0/1 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 1/1 [00:00<00:00,  1.86it/s]


In [8]:
embeddings

defaultdict(list,
            {4: [array([ 1.99677324e+00, -5.91030121e-01, -1.31713524e-01, -5.90262175e-01,
                      7.78515160e-01, -2.01714087e+00,  6.36825919e-01,  7.95911133e-01,
                      1.38128847e-01, -1.36370912e-01, -4.73926485e-01, -1.82835430e-01,
                     -9.81422424e-01,  1.41925782e-01, -7.55571544e-01,  1.87699571e-01,
                     -8.57385933e-01, -3.97630721e-01, -8.92586470e-01, -8.59309733e-02,
                      2.47310605e-02, -6.03079975e-01,  8.40921581e-01,  7.78481483e-01,
                     -6.70208037e-01,  7.11428702e-01, -9.72960413e-01,  3.85025144e-01,
                     -5.28060913e-01, -2.19213307e-01, -2.85363346e-01,  1.66318536e-01,
                      5.47599256e-01,  5.58065295e-01, -1.23867750e+00, -6.17281795e-01,
                      5.04537046e-01,  7.52454102e-01, -1.02123380e+00, -1.28704131e-01,
                      1.68685877e+00,  3.65311801e-01,  8.97850469e-02,  9.77435470e-01,


The next step is to calculate embeddings for the homonyms and their sentences that we got from the RAW-C dataset.

Make sure that your final output includes the word, the meaning id (M1_a, etc), the corresponding sentence and the embeddings at static layer and layers 4, 8, 12. You should maximally optimise this process by calculating in batches (again, check psycho-embeddings documentation), but keep in mind this might still take a while. First test your pipeline with a small number of inputs, and only run the full scale embedding extraction once you're positive the code works as expected.

When done, save the output in [pickle](https://docs.python.org/3/library/pickle.html) format (this is similar to json, but it can also handle np.arrays), so that you can easily load it later when needed and do not have to run it again. After pickle dumping (that's the word for saving it in pickle format), print it so that you are sure everything was saved correctly.

Then, check that your final data includes everything that you need by checking the entry "bank" and print the data pertaining to "bank".

In [9]:
items = []

meaning_cols = ["M1_a", "M1_b", "M2_a", "M2_b"]

for _, row in data.iterrows():
    for col in meaning_cols:
        if pd.notna(row[col]):
            items.append({
                "Word": row["Word"],   # use capital W everywhere
                "meaning_id": col,
                "sentence": row[col]
            })

print("Total items:", len(items))
print(items[:3])

Total items: 460
[{'Word': 'lamb', 'meaning_id': 'M1_a', 'sentence': 'They liked the marinated lamb.'}, {'Word': 'lamb', 'meaning_id': 'M1_b', 'sentence': 'They liked the grilled lamb.'}, {'Word': 'lamb', 'meaning_id': 'M2_a', 'sentence': 'They liked the cute lamb.'}]


In [10]:
# 1) Load final RAW-C file so we only keep stimuli that are in the final dataset
rawc = pd.read_csv("/content/ChickenHotWings/data/processed/raw-c.csv")

# Build the set of final (Word, String) pairs
final_pairs = (
    rawc[["word", "string"]]
    .drop_duplicates()
    .rename(columns={"word": "Word", "string": "String"})
)

# Use the already-created dataframe from your previous cell: `data`
final_data = (
    data.merge(final_pairs, on=["Word", "String"], how="inner")
    .reset_index(drop=True)
)

print("Final stimulus-level data:")
print(final_data.head())
print(f"Number of final stimuli: {len(final_data)}")

Final stimulus-level data:
      String       Word                            M1_a  \
0       lamb       lamb  They liked the marinated lamb.   
1       book       book     He had a best-selling book.   
2  breakfast  breakfast   They ate a pancake breakfast.   
3    chicken    chicken   They liked the juicy chicken.   
4      class      class      It was a perceptive class.   

                               M1_b                              M2_a  \
0      They liked the grilled lamb.         They liked the cute lamb.   
1            He had a popular book.              He had a heavy book.   
2  They ate a nutritious breakfast.      They ate a family breakfast.   
3   They liked the roasted chicken.  They liked the clucking chicken.   
4       It was a misbehaving class.            It was a boring class.   

                            M2_b  
0  They liked the friendly lamb.  
1   He had a leather-bound book.  
2   They ate a lonely breakfast.  
3    They liked the pet chicken.  
4   

In [11]:
# 2) Reshape so each row is one target word in one sentence
long_data = (
    final_data
    .melt(
        id_vars=["Word", "String"],
        value_vars=["M1_a", "M1_b", "M2_a", "M2_b"],
        var_name="meaning_id",
        value_name="sentence"
    )
    .reset_index(drop=True)
)

# Rename for clarity:
# Word   = lemma / regular form
# String = lexeme / surface form in the sentence
long_data = long_data.rename(columns={"Word": "lemma", "String": "word"})

print("\nSentence-level data to embed:")
print(long_data.head(8))
print(f"Number of sentence instances: {len(long_data)}")


Sentence-level data to embed:
          lemma          word meaning_id                            sentence
0          lamb          lamb       M1_a      They liked the marinated lamb.
1          book          book       M1_a         He had a best-selling book.
2     breakfast     breakfast       M1_a       They ate a pancake breakfast.
3       chicken       chicken       M1_a       They liked the juicy chicken.
4         class         class       M1_a          It was a perceptive class.
5  contribution  contribution       M1_a  He made a charitable contribution.
6        design        design       M1_a       They liked the floral design.
7        dinner        dinner       M1_a       They liked the turkey dinner.
Number of sentence instances: 448


In [12]:
# 3) Test with a small number of inputs
test_n = 8
test_output = model.embed(
    words=long_data["word"].head(test_n).tolist(),
    target_texts=long_data["sentence"].head(test_n).tolist(),
    layers_id=[4, 8, 12],
    batch_size=8,
    return_static=True
)

print("\nSmall test output keys:")
if isinstance(test_output, dict):
    print(list(test_output.keys()))
else:
    print(type(test_output), test_output)

Text tokenization:   0%|          | 0/8 [00:00<?, ? examples/s]

  0%|          | 0/1 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 1/1 [00:03<00:00,  3.72s/it]


Small test output keys:
[4, 8, 12, -1]


In [13]:
test_output

defaultdict(list,
            {4: [array([ 2.15850925e+00,  9.74830687e-01, -3.70509475e-01,  9.80445533e-04,
                     -1.09960401e+00,  7.99512982e-01,  1.29092586e+00,  5.18044591e-01,
                      2.29119569e-01, -1.08413458e-01, -3.07157159e-01, -6.46355629e-01,
                     -5.88293314e-01,  1.30274683e-01, -1.86863136e+00,  7.34583020e-01,
                     -6.29490256e-01,  5.90522408e-01, -8.80547404e-01, -1.16276167e-01,
                      3.36098880e-01, -1.50309578e-01, -1.75188705e-01,  2.16959089e-01,
                      4.69305545e-01,  8.25782239e-01, -4.95387197e-01,  3.02517027e-01,
                     -7.43336499e-01, -6.07794762e-01, -3.75814229e-01, -2.74741858e-01,
                      1.08879507e+00, -3.42119724e-01, -3.95872295e-01,  2.32038479e-02,
                     -1.15044475e+00,  7.67074585e-01, -6.82847619e-01, -9.19676721e-01,
                      7.65109777e-01, -2.45269847e+00,  3.05973113e-01,  1.43261933e+00,


In [14]:
# 4) Full embedding extraction
embed_output = model.embed(
    words=long_data["word"].tolist(),
    target_texts=long_data["sentence"].tolist(),
    layers_id=[4, 8, 12],
    batch_size=32,   # lower to 16 if Colab runs out of memory
    return_static=True
)


Text tokenization:   0%|          | 0/448 [00:00<?, ? examples/s]

 21%|██▏       | 3/14 [00:54<03:32, 19.36s/it]

KeyboardInterrupt: 

In [ ]:
# 5) Attach embeddings
result = long_data.copy()

if not isinstance(embed_output, dict):
    raise TypeError(f"Expected model.embed(...) to return a dict, got {type(embed_output)}")

for key, value in embed_output.items():
    key_str = str(key).lower()

    if key_str == "static":
        col_name = "embedding_static"
    else:
        digits = "".join(ch for ch in key_str if ch.isdigit())
        col_name = f"embedding_{digits}" if digits else f"embedding_{key_str}"

    arr = np.asarray(value)
    if arr.shape[0] != len(result):
        raise ValueError(f"{col_name} has {arr.shape[0]} rows, expected {len(result)}")

    result[col_name] = list(arr)
    print(f"Added {col_name}: {arr.shape}")

# keep the required columns in order
required_cols = [
    "lemma",          # regular form
    "word",           # surface form in sentence
    "meaning_id",
    "sentence",
    "embedding_1",    # static layer
    "embedding_4",
    "embedding_8",
    "embedding_12",
]

result = result[required_cols]


In [ ]:
result

In [ ]:
# 6) Save to pickle
pickle_path = "/content/rawc_homonym_embeddings.pkl"
with open(pickle_path, "wb") as f:
    pickle.dump(result, f)

In [ ]:

# 7) Load again and print to verify
with open(pickle_path, "rb") as f:
    loaded_result = pickle.load(f)

print("\nReloaded pickle preview:")
print(loaded_result.head())
print(f"\nSaved pickle path: {pickle_path}")
print(f"Total embedded rows: {len(loaded_result)}")

In [ ]:
# 8) Check and print the entry 'bank'
# Try lemma first, then surface form
bank_rows = loaded_result[
    (loaded_result["lemma"].astype(str).str.lower() == "bank") |
    (loaded_result["word"].astype(str).str.lower() == "bank")
]

print("\nData for 'bank':")
print(bank_rows)


In [ ]:
# show embedding shapes for bank
if len(bank_rows) > 0:
    for col in ["embedding_1", "embedding_4", "embedding_8", "embedding_12"]:
        print(f"{col} shapes:", bank_rows[col].apply(lambda x: np.asarray(x).shape).tolist())

# Task 2: Compute sense embeddings for the homonym dataset using WordNet [20 points]

Your next task is to fetch the definitions (glosses) of the homonyms, and compute an embedding for each gloss (each gloss is associated with a specific sense). We do that so we can later see whether the contextualised embeddings computed above represent the meaning of the homonym in context well (by comparing it to the sense embeddings). Figure 18.9 in [Jurafsky's and Martin's (2021) chapter 18](https://web.stanford.edu/~jurafsky/slp3/old_sep21/18.pdf) graphically illustrates this idea. Use this chapter for this part of the assignment, as it will come useful for you both theoretically and practically.

## Task 2.1: Fetch senses and glosses for a word [5 points]

First of all, you will have to figure out how [WordNet](https://www.nltk.org/howto/wordnet.html) works within the nltk package (hint: pay attention to what a synset is).

Install and import all the necessary components and define a function to extract the glosses of a word and create a dictionary with senses and glosses.

Then use the word "bat" to test that everything is working correctly: i.e., for "bat", you should be able to get its senses and the gloss for each of the sense (you will see that synsets might contain related words, but you only need the senses that contain the word of interest or derivates thereof; this should be specified in the function). Print the output for "bat".


In [15]:
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [16]:
#example synset of bat
wn.synsets('bat')

[Synset('bat.n.01'),
 Synset('bat.n.02'),
 Synset('squash_racket.n.01'),
 Synset('cricket_bat.n.01'),
 Synset('bat.n.05'),
 Synset('bat.v.01'),
 Synset('bat.v.02'),
 Synset('bat.v.03'),
 Synset('bat.v.04'),
 Synset('cream.v.02')]

In [17]:
def extract_glosses(word, dict_only=True):
  """
  Extracts glosses for a given word.
  Input: word --> string representing a word
         dict_only --> boolean indicating whether to ignore the print statements or not
  Returns: dictionary containing senses and glosses
  """
  glosses_dict = dict()
  #for each synset of the input word
  for synset in wn.synsets(word):
    if dict_only is False:
      print(synset)
      if len(synset.examples()) > 0:                    #if there are examples
        print('examples: ', synset.examples())          #print the examples
      #the lemma represents a specific sense of a specific word
      print('lemmas: ', [str(lemma.name()) for lemma in synset.lemmas()])     #print the lemma (always existing)
      print('gloss:', synset.definition(), '\n')        #print the gloss
    glosses_dict[synset] = synset.definition()
  return glosses_dict

In [18]:
extract_glosses(word='bat') # <-- add dict_only=False to view additional information

{Synset('bat.n.01'): 'nocturnal mouselike mammal with forelimbs modified to form membranous wings and anatomical adaptations for echolocation by which they navigate',
 Synset('bat.n.02'): '(baseball) a turn trying to get a hit',
 Synset('squash_racket.n.01'): 'a small racket with a long handle used for playing squash',
 Synset('cricket_bat.n.01'): 'the club used in playing cricket',
 Synset('bat.n.05'): 'a club used for hitting a ball in various games',
 Synset('bat.v.01'): 'strike with, or as if with a baseball bat',
 Synset('bat.v.02'): 'wink briefly',
 Synset('bat.v.03'): 'have a turn at bat',
 Synset('bat.v.04'): 'use a bat',
 Synset('cream.v.02'): 'beat thoroughly and conclusively in a competition or fight'}

## Task 2.2: Function to compute sense embeddings [10 points]

Now that you have a function to extract senses and glosses for a given word, write a function that takes a word and computes embeddings for each of the senses following the method explained in Jurafsky's and Martin's chapter. In this case, no need to calculate at different layers: you should use the last layer only. You should maximally optimise this function like before.

The output should include the sense, the gloss, and the embedding. Print the function's output when using the word "bank".


In [19]:
# Load BERT
#The paper uses BERT as well, uncased version since that was mentioned earlier in the assingment


tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")
model.eval() # Puts the model in "evaluation" mode, meaning feed-forward operation (more efficient)



loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/config.json
Model config BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

loading weights file model.safetensors from cache at /root/.cache/huggingfa

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [23]:
def compute_embeddings(word):

  """
  Input: word --> string representing a word
  Returns: dictionary: {the senses: {'gloss': gloss, 'embedding': embedding}} glosses + embeddings for the input word
  """

  computed_embeddings = {}

  #obtain glosses using the function we made earlier
  glosses = extract_glosses(word)

  # tokenize the glosses
  inputs = tokenizer(
      list(glosses.values()), # this takes the glosses as context
      return_tensors = 'pt',  # 'pt' -- returns to pytorch tensors
      padding = True,         # to make sure all glosses have the same length
      truncation = True)      # bool to make sure that long glosses are cut off

  # This evaluates BERT on our glosses, and fetch the hidden states of the network
  with torch.no_grad():
    outputs = model(**inputs)

  #mean pool
  embeddings = outputs.last_hidden_state.mean(dim = 1)

  #put embeddings in a dictionary: {sense: {gloss: gloss, embedding : embedding}}
  for i, sense in enumerate(glosses.keys()):
    gloss = glosses[sense]
    # make sure gloss is string
    if not isinstance(gloss, str):
        gloss = str(gloss)

    computed_embeddings[sense.name()] = {
        'gloss': glosses[sense],
        'embedding': embeddings[i].cpu().numpy().tolist()
    }
    # print("\nSense:", sense)
    # print("Gloss:", glosses[sense])
    # print("Embedding (first 5 dimensions):", [round(float(x), 4) for x in embeddings[i][:5]])

  return computed_embeddings


In [24]:
compute_embeddings('bank')

{'bank.n.01': {'gloss': 'sloping land (especially the slope beside a body of water)',
  'embedding': [-0.25153377652168274,
   -0.18591301143169403,
   0.22329390048980713,
   0.2396470308303833,
   0.07026815414428711,
   0.19687381386756897,
   0.010378596372902393,
   0.4470617175102234,
   -0.12880446016788483,
   -0.47140249609947205,
   -0.06577486544847488,
   0.12019544839859009,
   0.045769158750772476,
   -0.1562991589307785,
   -0.39957693219184875,
   0.27201735973358154,
   0.06552214175462723,
   -0.06854934245347977,
   -0.0425080806016922,
   0.427565336227417,
   -0.002907203044742346,
   0.07866226136684418,
   -0.09833826124668121,
   -0.05942057818174362,
   0.24999448657035828,
   -0.10619592666625977,
   0.23294174671173096,
   0.48890262842178345,
   -0.12187793105840683,
   0.012918747961521149,
   0.1611609309911728,
   -0.22895966470241547,
   0.15281616151332855,
   -0.004465695470571518,
   -0.061719439923763275,
   -0.4297211170196533,
   0.1342438459396362

## Task 2.3: Compute sense embeddings for the RAW-C stimuli [5 points]

Now, use the function you defined above to compute sense embeddings for the RAW-C stimuli and pickle dump it too.

As above, the information that should be there for each word is: the sense, the gloss, the embedding at the last layer. Again, you can think of which structure to use best, but keep in mind that we will have to compare these to the CWE calculated in task 1, so it is good to think of a similar structure that is easily comparable.

Make sure that the number of stimuli matches the number of stimuli in the final RAW-C dataset.

In [29]:
rawc_stimuli = rawc['word'].tolist() #converts the 'word' column to a list
print("Number of stimuli:", len(rawc_stimuli))  #print the number of stimuli: 672
print("Unique words in rawc: ", len(set(rawc_stimuli)))


#Compute all the embeddings for the words in the raw-c file
rawc_embeddings = {}
for word in final_data["Word"][:10]: #at this moment you only do 10 of the words, change this for the full stimuli
  rawc_embeddings[word] = compute_embeddings(word)

#Save to pickle
pickle_path = "/content/rawc_sense_embeddings.pkl"
with open(pickle_path, 'wb') as f:
    pickle.dump(rawc_embeddings, f)

Number of stimuli: 672
Unique words in rawc:  112


In [26]:
# Load again and print to verify
with open(pickle_path, "rb") as f:
    loaded_result = pickle.load(f)

print("\nReloaded pickle preview:")
print(loaded_result)
print(f"\nSaved pickle path: {pickle_path}")
print(f"Total embedded rows: {len(loaded_result)}")


Reloaded pickle preview:
{'lamb': {'lamb.n.01': {'gloss': 'young sheep', 'embedding': [-0.2642497420310974, -0.009659533388912678, -0.36188244819641113, -0.01884431578218937, 0.33446353673934937, 0.3761126399040222, 0.2285119891166687, 0.10680350661277771, -0.49806585907936096, -0.2690105438232422, -0.08893564343452454, 0.038500573486089706, -0.09014598280191422, -0.23342439532279968, -0.5770708918571472, -0.06548377126455307, 0.03878745436668396, 0.2601103186607361, 0.25110185146331787, -0.20590585470199585, 0.21316519379615784, -0.4151690900325775, 0.08722519874572754, -0.3168366551399231, 0.25512582063674927, -0.054153766483068466, 0.17399531602859497, -0.06707529723644257, -0.2986787259578705, -0.018095165491104126, -0.24174657464027405, -0.28948211669921875, 0.5869687795639038, 0.6871744394302368, -0.03778769075870514, -0.26510265469551086, 0.29596981406211853, -0.20274019241333008, 0.04530050978064537, 0.013461570255458355, 0.23311519622802734, 0.059400223195552826, 0.0178460758

# Task 3: Compute and explore similarity between homonym CWEs and sense embeddings [35 points]

You now have the homonym CWEs computed in task 1, and the sense embeddings computed in task 2. The next step is to calculate cosine similarities between each CWE for each homonym (at the selected layer!) and each sense embedding for that homonym.

For instance, say for the word "bat" with meaning M1_a, you have its CWE at the static layer and at layers 4, 8, 12 and 7 senses: here, you will end up with 16 cosine similarities (take each CWE and compute its similarity to each of the sense embeddings). We then want to see which sense meaning is the closest to each CWE, and do some qualitative explorations with that.

## Task 3.1: Compute the cosine similarity between all the CWEs and the sense embeddings [8 points]

This task is not trivial with regards to how much information you have and how to structure the data (this is why it's also important to think of data structures in the earlier parts of the assignment), so take some time to think how to best breakdown this task. Test each step/function if you have multiple. Pickle dump your final output so that it is easily retrievable for later. At the end, print an example of the entry "bank".

For cosine similarity, the cdist function from scipy.spatial.distance seems the most efficient, but you are free to use any of your liking (hint: pay attention to the shape of your embeddings and to similarity vs distance. You will need the similarity).

In [ ]:
#your code here

## Task 3.2: Quantitative and qualitative explorations the relationship between homonym embeddings and dominant senses

Now, we can look into how the CWEs in different meanings and layers relate to the different senses of a homonym. We'll focus on the dominant sense in WordNet, see below for more details. This section includes both code blocks and reflection questions.

### Dominant senses in WordNet and top senses across layers (focus on static layer) [8 points]

Embeddings at the static layer do not take into account context, so intuitively they should capture the 'average' meaning, maybe the most common/dominant. We can test this by looking at the most similar sense and seeing if that matches that most common/dominant sense in the synset.

Keep in mind that synsets mark more common/dominant senses with numbering: so n.01 will be the most common noun; v.01 the most common verb, etc. If that is not available, the most common meaning will be the next number (e.g., n.02). You have to take that into account when you extract the top sense, so first extract information about which are the most dominant senses for each word across all the parts of speech: for example, "bat" might have as its two most common senses bat.n.01 and bat.v.02 (because v.01 might not be available; this is just an example). Some words might only have one part of speech in their synset, some more. Print your results.

In [ ]:
#your code here

Then, extract the top similarity of homonyms to the senses at all the layers you have available. While we are interested in the static layer for checking dominant senses, it is also interesting to look into other layers to see whether adding context will refine the captured meaning.


In [ ]:
#your code here

Let's check an example from our results.

Out of all the similarities of 'bank' to all its senses at all the layers, which one is the highest? Print your results for that entry and reflect below.

In [ ]:
#your code here

### Does the static layer capture the most dominant meaning, according to WordNet (and according to you)? [2 point]

%your answer here

### Across other layers and meanings, which layer seems to capture the meaning of bank across meanings best, and why do you make this conclusion? [2 points]

%your answer here

### Checking matches and mismatches with the dominant sense [5 points]

Now, let's quantitatively check if the static layer actually captures the most dominant sense (any POS). You should end up with two data structures: matches (when the most similar sense is one of the dominant senses) and mismatches (when the most similar sense is not one of the dominant sense). Do that also for the other layers to compare. Print the percentage of matches and mismatches per layer.



In [ ]:
#your code here

Now, print the matches and mismatches for the static layer only.

In [ ]:
#your code here

### Do BERT's static embeddings capture the most dominant sense in WordNet? [2 point]

%your answer here

### Do the percentages of matches and mismatches throughout the layers make sense to you or is it different than what you expected? [2 points]

%your answer here

### For the **static layer**, are there any words that seem to particularly deviate from the dominant meaning? If so, which and why could that be? [3 points]

%your answer here

### Do you think the corpus on which BERT is trained might reflect different meaning dominance than for WordNet's senses? If so/not, why? [3 points]

%your answer here

# Task 4: Partially replicate Trott & Bergen's experiment [20 points]

Now comes the time to partially replicate the RAW-C experiment, by seeing whether different layers of BERT capture meanings more or less similarly to humans. At the end you will have to wrap up with a brief comment on which layer seems to capture meanings best and how that connects to explorations in the previous section.

## Task 4.1: Create a dataframe with cosine similarities between sentences at different layers [7 points]

You should now use the embeddings at the different layers that you computed to calculate similarities between each context: M1a, M1b, M2a, M2b. You will have to have all combinations, so for each string in the RAW-C dataframe, you'll have: M1a vs M1b, M1a vs M2a, M1a vs M2b, M1b vs M2a, M1b vs M2b, M2a vs M2b.

Bear in mind that your final dataframe should include: the word, the string as it appears in the sentence, cosine similarity at layers 4, layer 8, layer 12, the version being compared (is it M1a vs M1b or M1a vs M2a?) and the mean relatadness given by humans (hint: the repo you cloned will come useful here, both in terms of code and data). Print the head of the dataframe to check everything is in order, and check also that the number of stimuli match with your number across the assignment (starting from task 1).

In [ ]:
#your code here

## Task 4.2: Correlate with human judgements and visualise [8 points]

First, correlate the cosine similarities at the different layers to the mean human relatedness judgements. Use the same correlation metric used by Trott & Bergen.

In [ ]:
#your code here

Next, visualise your results. You want to see the correlation between BERT embeddings and human judgements per layer, but what would also be interesting is to include the meaning contrasts (such as M1_a_M1_b, etc), so that we can see how those play out per layer.

In [ ]:
#your code here

### Reflect on the correlations and on the visualisations. What can you observe and infer in terms of which layer(s) might be capturing meaning best? Is there one way to determine that (i.e., what does 'capturing meanings' mean?)? Contrast and compare the layers. [5 points]

%your answer here



